In [1]:
# 1. Install all dependencies (now with the Numpy fix built-in!)
%pip install "numpy<2" medspacy spacy openai pandas textstat rouge-score
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# 2. Download the base SpaCy model directly through Python
import spacy
spacy.cli.download("en_core_web_sm")

print("All installations complete and in the correct environment!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz (119.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All installations complete and in the correct environment!


In [2]:
# Install all the required libraries into the new Python 3.11 environment
%pip install pandas openai textstat rouge-score

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Install required libraries if you haven't already:
# !pip install medspacy spacy openai pandas textstat rouge-score
# !python -m spacy download en_core_web_sm


import pandas as pd
import medspacy
from openai import OpenAI
import textstat
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

# 1. Load MedSpaCy for the Extraction Stage
print("Loading MedSpaCy model...")
nlp = medspacy.load()

# 2. Initialize LLM Client (Using OpenAI format, works for local Llama-3 via Ollama too)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # This can be any string, Ollama doesn't check it
)
print("Environment setup complete.")

Loading MedSpaCy model...
Environment setup complete.


In [2]:
import pandas as pd

# Using "r" before the string tells Python to read the Windows backslashes correctly
notes_path = r"c:\Users\kirth\OneDrive\Desktop\Study\Sem-3\NLP\A3\NLPA3\data\NOTEEVENTS_sorted.csv" 
patients_path = r"c:\Users\kirth\OneDrive\Desktop\Study\Sem-3\NLP\A3\NLPA3\data\PATIENTS_sorted.csv"

# --- OPTION A: Using real MIMIC-III files ---
print("Attempting to load data from exact path...")
notes = pd.read_csv(notes_path, nrows=5000) 
patients = pd.read_csv(patients_path)

df = pd.merge(notes, patients[['SUBJECT_ID', 'DOB']], on='SUBJECT_ID')
elderly_notes = df[df['CATEGORY'] == 'Discharge summary'].head(100)
test_note = elderly_notes.iloc[0]['TEXT']

print("Success! Data loaded.")

# # --- OPTION B: Synthetic Note for immediate testing ---
# test_note = """
# Discharge Diagnosis: Community-acquired pneumonia. Right lower zone consolidation.
# Medications on Discharge: 
# 1. Amoxicillin 500mg PO TID for 7 days.
# 2. Ibuprofen 400mg PO PRN for fever or pain.
# Follow-up: Patient was tachycardic and tachypnoeic on admission. Return to ED if HR > 110 or experiencing severe dyspnea.
# """
# print("Data loaded successfully.")

Attempting to load data from exact path...
Success! Data loaded.


In [3]:
import sys
import subprocess

print(f"Jupyter is currently using this Python: {sys.executable}")
print("Forcing installation into this exact location...")

# This forces pip to run inside the exact environment Jupyter is using
subprocess.check_call([
    sys.executable, 
    "-m", 
    "pip", 
    "install", 
    "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz"
])

print("Installation complete! Trying to import...")
import en_ner_bc5cdr_md
print("SUCCESS! The model is loaded and ready.")

Jupyter is currently using this Python: c:\Users\kirth\AppData\Local\Programs\Python\Python311\python.exe
Forcing installation into this exact location...
Installation complete! Trying to import...
SUCCESS! The model is loaded and ready.


In [12]:
import medspacy
import en_ner_bc5cdr_md # We import the model directly as a Python library

print("Loading Clinical NLP Model...")

# 1. Load the base clinical model directly from the library
base_clinical_nlp = en_ner_bc5cdr_md.load()

# 2. Pass that loaded model into MedSpaCy
nlp = medspacy.load(base_clinical_nlp)

def extract_clinical_facts(raw_text):
    """Stage 1: Upgraded ScispaCy/MedSpaCy NER Extraction"""
    doc = nlp(raw_text)
    
    # The clinical model uses 'CHEMICAL' (for drugs) and 'DISEASE'
    target_labels = ['CHEMICAL', 'DISEASE']
    
    # Extract entities and remove duplicates
    entities = list(set([ent.text for ent in doc.ents if ent.label_ in target_labels]))
    
    # Join them into a comma-separated string
    return ", ".join(entities)
def generate_patient_summary(raw_text, locked_facts):
    """Stage 2: Ultra-Simple Generation for 8th Grade Target"""
    
    system_prompt = f"""You are a caregiver for an elderly patient. 
    Your goal is to explain their medical note using very simple, short words. 
    Target a 6th to 8th-grade reading level.

    STRICT RULES:
    1. Use ONLY these headings: "Diagnosis:", "Medication:", "Red Flags:".
    2. Use short sentences. 
    3. Instead of 'Pneumonia', say 'Lung infection (Pneumonia)'.
    4. Instead of 'Medication', say 'Meds' or 'Pills'.
    5. Instead of 'Physician', say 'Doctor'.
    6. Mention these exact facts: {locked_facts}
    7. No Latin abbreviations like PO, TID, or PRN. Say 'by mouth' or '3 times a day'."""

    response = client.chat.completions.create(
        model="gpt-4o", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Simplify this: {raw_text}"}
        ],
        temperature=0.1 # Low temperature keeps it focused on the facts
    )
    return response.choices[0].message.content

Loading Clinical NLP Model...


In [13]:
def evaluate_pipeline(original_text, simplified_text, locked_facts):
    """Calculates Readability and Fidelity metrics"""
    
    # 1. Readability: Flesch-Kincaid Grade Level (Target: <= 8.0)
    grade_level = textstat.flesch_kincaid_grade(simplified_text)
    
    # 2. Clinical Fidelity: ROUGE-L (Comparing locked facts against final output)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    # We test if the simplified text retained the extracted facts
    rouge_scores = scorer.score(locked_facts, simplified_text)
    
    return {
        "Flesch-Kincaid Grade": grade_level,
        "ROUGE-L F1": rouge_scores['rougeL'].fmeasure
    }

In [14]:
import medspacy
import en_ner_bc5cdr_md 

print("Loading Clinical NLP Model...")

# THE FIX: Disable the base model's parser so it doesn't fight with MedSpaCy
base_clinical_nlp = en_ner_bc5cdr_md.load(disable=["parser"])

# 2. Pass that loaded model into MedSpaCy
nlp = medspacy.load(base_clinical_nlp)

def extract_clinical_facts(raw_text):
    """Stage 1: Upgraded ScispaCy/MedSpaCy NER Extraction"""
    doc = nlp(raw_text)
    
    # The clinical model uses 'CHEMICAL' (for drugs) and 'DISEASE'
    target_labels = ['CHEMICAL', 'DISEASE']
    
    # Extract entities and remove duplicates
    entities = list(set([ent.text for ent in doc.ents if ent.label_ in target_labels]))
    
    # Join them into a comma-separated string
    return ", ".join(entities)

# Test it right away!
test_note = """
Discharge Diagnosis: Community-acquired pneumonia. Right lower zone consolidation.
Medications on Discharge: Amoxicillin 500mg PO TID for 7 days. Ibuprofen 400mg PO PRN for fever or pain.
"""
print("EXTRACTED FACTS:", extract_clinical_facts(test_note))

Loading Clinical NLP Model...


2026-05-05 00:21:25.466 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 1 'Discharge' marked as sentence start (span begin)
2026-05-05 00:21:25.467 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 9 'Right' marked as sentence start (span end next token)
2026-05-05 00:21:25.467 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 9 'Right' marked as sentence start (span begin)
2026-05-05 00:21:25.467 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 14 '
' marked as sentence start (span end whitespace)
2026-05-05 00:21:25.467 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] GAP DETECTED: tokens 14-14 (idx 83-83) between spans 83-84
2026-05-05 00:21:25.467 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 14 '

EXTRACTED FACTS: pain, Amoxicillin, pneumonia, fever, Ibuprofen


In [15]:
import textstat
from rouge_score import rouge_scorer

# --- STAGE 1: EXTRACTION (Already defined, but here for completeness) ---
def extract_clinical_facts(raw_text):
    doc = nlp(raw_text)
    target_labels = ['CHEMICAL', 'DISEASE']
    entities = list(set([ent.text for ent in doc.ents if ent.label_ in target_labels]))
    return ", ".join(entities)

# --- STAGE 2: GENERATION ---
def generate_patient_summary(raw_text, locked_facts):
    system_prompt = f"""You are a caregiver. Translate medical notes into 8th-grade level instructions.
    RULES: 
    1. Use headings: "Diagnosis:", "Medication:", "Red Flags:".
    2. No Latin (e.g., change 'b.i.d' to 'twice a day').
    3. Include these facts: {locked_facts}"""
    
    response = client.chat.completions.create(
        model="llama3", # or "llama3"
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": raw_text}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

# --- STAGE 3: EVALUATION ---
def evaluate_summary(summary, original_facts):
    # 1. Calculate Reading Level
    grade_level = textstat.flesch_kincaid_grade(summary)
    
    # 2. Calculate Fact Retention (ROUGE-L)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = scorer.score(original_facts, summary)
    
    return grade_level, scores['rougeL'].fmeasure

# --- EXECUTION ---
test_note = """
Discharge Diagnosis: Community-acquired pneumonia. Right lower zone consolidation.
Medications on Discharge: Amoxicillin 500mg PO TID for 7 days. Ibuprofen 400mg PO PRN for fever or pain.
"""

print("--- STAGE 1: EXTRACTING FACTS ---")
facts = extract_clinical_facts(test_note)
print(f"Locked Facts: {facts}")

print("\n--- STAGE 2: GENERATING SUMMARY ---")
summary = generate_patient_summary(test_note, facts)
print(f"Final Summary:\n{summary}")

print("\n--- STAGE 3: EVALUATION METRICS ---")
grade, rouge = evaluate_summary(summary, facts)
print(f"Reading Level (Flesch-Kincaid): {grade} (Target: <= 8.0)")
print(f"Fact Retention (ROUGE-L): {rouge:.2f} (Target: > 0.0)")

2026-05-05 00:21:32.504 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 1 'Discharge' marked as sentence start (span begin)
2026-05-05 00:21:32.504 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 9 'Right' marked as sentence start (span end next token)
2026-05-05 00:21:32.506 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 9 'Right' marked as sentence start (span begin)
2026-05-05 00:21:32.506 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 14 '
' marked as sentence start (span end whitespace)
2026-05-05 00:21:32.506 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] GAP DETECTED: tokens 14-14 (idx 83-83) between spans 83-84
2026-05-05 00:21:32.507 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 14 '

--- STAGE 1: EXTRACTING FACTS ---
Locked Facts: pain, Amoxicillin, pneumonia, fever, Ibuprofen

--- STAGE 2: GENERATING SUMMARY ---
Final Summary:
**Your Care Plan**

**Diagnosis:** You have a type of lung infection called community-acquired pneumonia, which affects the right lower part of your lungs.

**Medication:**

* Take Amoxicillin (an antibiotic) three times a day (TID) for 7 days. This medicine will help clear up the infection in your lungs.
* If you have a fever or pain, take Ibuprofen (a pain reliever) as needed (PRN). This medicine can help reduce your fever and ease any discomfort.

**Red Flags:**

* Watch out for signs of worsening pneumonia, such as:
	+ Increasing cough
	+ More severe chest pain
	+ Fever that doesn't go away or gets worse
	+ Shortness of breath

Remember to follow these instructions carefully and take your medication as directed. If you have any concerns or notice any changes in your symptoms, be sure to reach out to your healthcare provider.

--- STAGE 3